# Day 3 - Conversational AI - aka Chatbot!

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

c:\Users\dell\Desktop\llm_engineering_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

if not api_key:
    print("No API key was found - please check your .env file!")
elif not api_key.startswith("sk-or-"):
    print("API key found but it doesn't look like an OpenRoute  r key (should start with sk-or-)")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [3]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
    
)
MODEL = 'gpt-4.1-mini'

In [4]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant"

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [8]:
import gradio as gr
print(gr.__version__)

6.13.0


In [5]:
def chat(message, history):
    return "bananas"

In [6]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [7]:
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [8]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


## OK! Let's write a slightly better chat callback!

In [9]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [10]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [11]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = client.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [12]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


## OK let's keep going!

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [13]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [14]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [15]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [16]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


In [17]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = client.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [18]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [22]:
system_message = """
أنت مساعد مبيعات ذكي لمحل اسمه "إيكون" لبيع البرفانات.

المحل يبيع برفانات حريمي ورجالي بأسعار مختلفة.

---

## البرفانات الحريمي:

### اسبلاش:
- صغير: 45 جنيه  
- كبير نوع 1: 75 جنيه  
- كبير نوع 2: 120 جنيه  

### برفانات حريمي أخرى:
- كايلي:
  - صغير: 120 جنيه  
  - كبير: 170 جنيه  

- برادا: 160 جنيه  
- the body shop: 110 جنيه  

---

## البرفانات الرجالي:
- سوفاج: 160 جنيه  
- بيرساجي: 160 جنيه  

---

## القواعد:
- رد فقط بناءً على المنتجات المذكورة أعلاه  
- لو المنتج غير موجود، قول إنه غير متوفر بشكل لبق  
- حاول تساعد العميل يختار أفضل منتج حسب احتياجه  
- كن ودود ومساعد في البيع  
"""

In [23]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    relevant_system_message = system_message

    messages = (
        [{"role": "system", "content": relevant_system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [24]:
gr.ChatInterface(fn=chat).launch(share=True)

* Running on local URL:  http://127.0.0.1:7882
* Running on public URL: https://93d6153581f9709c38.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
